The goal of this notebook is to train some adversarial patches on the ImageNet data set and visualize what these patches look like, as well as explore the effectiveness of patches once we alter certain features.

This notebook is largely modelled after: https://github.com/AIPI-590-XAI/Duke-AI-XAI/blob/main/adversarial-ai-example-notebooks/adversarial_attacks_patches.ipynb

I am writing my own version of tihs notebook to build a stronger understanding of this work.

In [ ]:
# Install necessary packages if they do not exist.
# %pip install --upgrade numpy
# %pip install numpy matplotlib torch torchvision
# %pip install --upgrade pip

In [ ]:
import os
import numpy as np
import math

import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

import torchvision
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import datasets, transforms

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")


There exist many kinds of models that torchvision offers. In terms of image classification, ResNet50 is mainstream. Choosing default pre-trained weights for ResNet50 forces the most up-to-date version.

In [ ]:
pretrained_model = torchvision.models.resnet50(weights = ResNet50_Weights.DEFAULT)
pretrained_model.to(device) # moves model to device

for p in pretrained_model.parameters():
    p.requires_grad = False

We need to normalize input. Color channel intensity is measured by z scores as opposed to their actual values. 

Numbers for mean and standard deviation can be found here: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html

In [ ]:
# Mean and Std from ImageNet
NORM_MEAN = np.array([0.485, 0.456, 0.406])
NORM_STD = np.array([0.229, 0.224, 0.225])


plain_transforms = transforms.Compose([
    transforms.ToTensor(), # Maps RGB to (0,1) as opposed to (0,255)
    transforms.Normalize(mean=NORM_MEAN,
                         std=NORM_STD) # 
])



We first want to evaluate how our model performs on unperturbed images. TinyImageNet contains 200 separate labels, each with 5 images which are drawn from ImageNet. ResNet50's pretrained weights are trained on IMAGENET1K_V2 (as of October 13th, 2026)

In [ ]:
imagenet_path = "./data/TinyImageNet"

dataset = datasets.ImageFolder(
    root = imagenet_path,
    transform = plain_transforms
    )

data_loader = DataLoader(
    dataset,
    batch_size = 32,
    shuffle = True,
    drop_last = False
)

print("Images:", len(dataset))
print("Classes:", len(dataset.classes))

images, labels = next(iter(data_loader))
print("Batch shape:", images.shape)

Now let's evaluate the model:

In [ ]:
def evaluate_model(model, data_loader, device):
    model.eval()
    correct_top1 = 0
    correct_top5 = 0
    total = 0


    with torch.no_grad():
        for images, labels in data_loader:
            # move image and label to GPU
            images = images.to(device)
            labels = labels.to(device)

            # inputs a image batch of 32. Returns [batchsize, 1000].
            # Each entry in 1000 represents the logits that our model attributes.
            scores = model(images) 

            # index of maximum value along horizontal axis, which is logits
            predictions = scores.argmax(dim = 1) 
            # item() returns the number into a python integer as opposed to a tensor num
            correct_top1 += (predictions == labels).sum().item() 

            # finds top 5 scoring indices
            top5_predictions = scores.topk(5, dim=1).indices 
            correct_top5 += (
                top5_predictions == labels.unsqueeze(dim = 1)
            ).any(dim = 1).sum().item()

            total += labels.size(0)

    return correct_top1 / total, correct_top5 / total

top1, top5 = evaluate_model(pretrained_model, data_loader, device)
error1 = 1 - top1
error5 = 1 - top5
print(f"Top-1 Accuracy {top1:.2%}")
print(f"Top-5 Accuracy {top5:.2%}")

print(f"Top-1 Error {error1:.2%}")
print(f"Top-5 Error {error5:.2%}")

Let's show some images to see what our model is doing before we begin training some patches.

In [ ]:
from matplotlib.ticker import PercentFormatter

class_names = ResNet50_Weights.DEFAULT.meta["categories"]

def show_predictions(model, data_loader, device, num_images=4):
    model.eval()
    images, labels = next(iter(data_loader))
    num_images = min(num_images, images.size(0)) # displays as many images as pos

    # get the images and labels, move to cpu since matplotlib works on cpu
    images = images[:num_images].cpu()
    labels = labels[:num_images].cpu()

    # disable tracking of how much our output changes
    with torch.no_grad():
        # move image to device, then process model prediction. shape num_images x num_labels
        logits = model(images.to(device))
        probabilities = logits.softmax(dim=1)
        top_probs, top_indices = probabilities.topk(5, dim=1)

        # moves our work back to cpu
        top_probs = top_probs.cpu()
        top_indices = top_indices.cpu()

        # what exactly is this doing?
        mean = images.new_tensor(NORM_MEAN).view(1,3,1,1)
        std = images.new_tensor(NORM_STD).view(1,3,1,1)

        display_images = (images * std + mean).clamp(0, 1) # undo normalization constants

        fix, axes = plt.subplots(
            nrows = num_images,
            ncols = 2,
            figsize= (12, 3 * num_images),
            squeeze = False,
            constrained_layout = True
        )

        for i in range(num_images):
            image_ax, graph_ax = axes[i]
            image_ax.imshow(display_images[i].permute(1,2,0).numpy()) # reorder dimensions
            true_label = data_loader.dataset.classes[labels[i].item()]
            predicted_label = class_names[top_indices[i, 0].item()]

            image_ax.set_title(
                f"Dataset Label: {true_label}\n"
                f"Prediction: {predicted_label}" 
            )

            image_ax.axis("off")

            # look at the corresponding name for each prediction
            names = [class_names[index.item()] for index in top_indices[i]]
            values = top_probs[i].numpy()
            graph_ax.barh(range(5), values, color="steelblue") # horizontal bars
            graph_ax.set_yticks(range(5), labels=names)
            graph_ax.invert_yaxis() # 0 first
            graph_ax.set_xlim(0, 1)
            graph_ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

            graph_ax.set_xlabel("Predicted probability")
            graph_ax.set_title("Top five predictions")

        plt.show()



In [ ]:
show_predictions(pretrained_model, data_loader, device, num_images=4)